In [1]:
# =============================================================================
# Zelle 01 – Setup & Imports
# =============================================================================
# Ziel: Umgebung fuer die synthetische Datengenerierung (Modell A) vorbereiten.
# Reproduzierbarkeit ueber festen Seed sichergestellt (wie in Vorprojekten).
# Hinweis: Alle Firmen-/Produktbezuege in diesem Projekt sind fiktiv
# (Extruder GmbH).
# =============================================================================

import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from viz_config import apply_store44_style, save_figure, COLOR_GOLD, COLOR_BLUE, COLOR_GREEN

# Reproduzierbarkeit
SEED = 42
rng = np.random.default_rng(SEED)

# Store44-Stil aktivieren (gilt ab jetzt fuer alle Plots in diesem Notebook)
apply_store44_style()

# Zielpfad fuer Rohdaten
DATA_RAW_PATH = "../data/raw/model_a_raw.csv"

print(f"Seed gesetzt: {SEED}")
print(f"numpy: {np.__version__}, pandas: {pd.__version__}")
print("Store44-Stil aktiviert.")

Seed gesetzt: 42
numpy: 2.4.6, pandas: 3.0.5
Store44-Stil aktiviert.


In [2]:
# =============================================================================
# Zelle 02 – Latente Auftragsgroesse (Rohr_DN) + X_A-Basisvariablen
# =============================================================================
# KORREKTUR 1 (Root-Cause-Fix, Mechanismus): urspruengliche Version nutzte
# durchgaengig Vakuum als Ausformungsmechanismus. Recherche ergab: Vakuum
# wird nur bei Nennweiten > 200mm eingesetzt, bei kleineren Nennweiten
# erfolgt die Ausformung ueber Formluft/Ueberdruck (Quelle: Wikipedia
# "Corrugator (Kunststoffverarbeitung)", DeWiki "Corrugator"). DN-Verteilung
# um Nennweiten > 200mm erweitert, damit beide Mechanismen im Datensatz
# vorkommen. Konkrete mbar-Werte bleiben unbelegte Annahmen (keine
# oeffentliche Quelle mit Zahlenwerten gefunden) - nur der MECHANISMUS
# (Vorzeichen/Schwelle bei DN200) ist quellenbasiert.
#
# KORREKTUR 2 (Root-Cause-Fix, Saettigung): systematischer Check aller
# DN-abhaengigen Formeln (ausgeloest durch EDA-Vertiefung Zelle 08b/8c)
# deckte auf, dass wandstaerke_ideal (72% Clipping), massedurchsatz (100%
# Clipping) und abzugsgeschwindigkeit (50% Clipping) bei DN>200 vollstaendig
# gesaettigt waren - die quadratische Skalierung (DN/100)^2 war nur fuer den
# urspruenglichen Bereich DN<=200 kalibriert. Alle drei Formeln auf lineare
# DN-Abhaengigkeit umgestellt und fuer den vollen Bereich (DN 50-315) neu
# kalibriert. querschnitt_proxy dadurch nicht mehr benoetigt, entfernt.
# =============================================================================

N = 700

dn_werte = np.array([50, 63, 75, 90, 110, 125, 160, 200, 250, 315])
dn_gewichte_basis = np.array([0.10, 0.14, 0.16, 0.18, 0.18, 0.12, 0.08, 0.04])
dn_gewichte = np.concatenate([dn_gewichte_basis * 0.92, [0.05, 0.03]])
rohr_dn = rng.choice(dn_werte, size=N, p=dn_gewichte)

# Kalibriermechanismus strukturell aus DN abgeleitet (nicht zufaellig)
kalibriermechanismus = np.where(rohr_dn > 200, "Vakuum", "Formluft")

# --- Zielwandstaerke (latent, nicht X_A): linear in DN, kalibriert auf ---
# --- vollen Bereich DN 50-315 (KORREKTUR 2) ---
wandstaerke_basis = 1.004 + rohr_dn * 0.007925
wandstaerke_ideal = np.clip(wandstaerke_basis + rng.normal(0, 0.13, N), 1.0, 4.0)

mfr_charge = np.clip(rng.normal(0.7, 0.15, N), 0.3, 1.2)

# --- Massedurchsatz: linear statt quadratisch in DN (KORREKTUR 2) ---
massedurchsatz = np.clip(10 + rohr_dn * 0.42 + rng.normal(0, 4, N), 5, 150)

sqrt_md = np.sqrt(massedurchsatz)
slope_dz = (75 - 15) / (sqrt_md.max() - sqrt_md.min())
intercept_dz = 15 - slope_dz * sqrt_md.min()
schneckendrehzahl = np.clip(intercept_dz + slope_dz * sqrt_md + rng.normal(0, 2.5, N), 10, 80)

massetemperatur = np.clip(215 - (mfr_charge - 0.7) * 8 + rng.normal(0, 4, N), 180, 230)

die_swell_faktor = rng.normal(0.85, 0.04, N)
duesenspalt = np.clip(wandstaerke_ideal * die_swell_faktor + rng.normal(0, 0.08, N), 0.5, 6.0)

# --- Abzugsgeschwindigkeit: linear statt quadratisch in DN (KORREKTUR 2) ---
abzugsgeschwindigkeit = np.clip(-0.67 + 4.34 * (rohr_dn / 100) + rng.normal(0, 0.4, N), 0.5, 15)

viskositaets_proxy = 1 / mfr_charge
md_norm = massedurchsatz / massedurchsatz.mean()
visk_norm = viskositaets_proxy / viskositaets_proxy.mean()
ds_norm = duesenspalt / duesenspalt.mean()
massedruck = np.clip(150 * md_norm * visk_norm / (ds_norm ** 1.5) * rng.normal(1.0, 0.08, N), 50, 300)

# --- Kalibrierdruck: Formluft (Ueberdruck, positiv) oder Vakuum (negativ), ---
# --- abhaengig vom strukturell durch DN bestimmten Mechanismus ---
formluft_druck = np.clip(50 + rohr_dn * 1.2 + rng.normal(0, 20, N), 20, 400)
vakuum_druck = np.clip(-150 - rohr_dn * 2.2 + rng.normal(0, 25, N), -900, -100)
kalibrierdruck_mbar = np.where(kalibriermechanismus == "Vakuum", vakuum_druck, formluft_druck)

# Normierte Prozessdruck-Staerke (0-1), mechanismus-unabhaengig - wird in
# Zelle 03/04 fuer Ausformungsguete verwendet
druck_norm = np.where(
    kalibriermechanismus == "Vakuum",
    np.clip((np.abs(vakuum_druck) - 100) / 800, 0, 1),
    np.clip((formluft_druck - 20) / 380, 0, 1),
)

kuehlwassertemperatur = np.clip(rng.normal(16, 4, N), 8, 25)

df = pd.DataFrame({
    "rohr_dn_latent": rohr_dn,
    "wandstaerke_ideal_latent": wandstaerke_ideal,
    "schneckendrehzahl": schneckendrehzahl,
    "massedurchsatz": massedurchsatz,
    "massetemperatur": massetemperatur,
    "massedruck": massedruck,
    "duesenspalt": duesenspalt,
    "abzugsgeschwindigkeit": abzugsgeschwindigkeit,
    "kalibriermechanismus": kalibriermechanismus,
    "kalibrierdruck_mbar": kalibrierdruck_mbar,
    "kuehlwassertemperatur": kuehlwassertemperatur,
    "mfr_charge": mfr_charge,
})

print(f"Shape: {df.shape}")
print(f"Anteil Vakuum-Mechanismus: {(kalibriermechanismus=='Vakuum').mean()*100:.1f}%")
print(f"Anteil Formluft-Mechanismus: {(kalibriermechanismus=='Formluft').mean()*100:.1f}%")

# --- Clipping-Rate pro Spalte pruefen (Qualitaetssicherung) ---
for col, (lo, hi) in {
    "schneckendrehzahl": (10, 80), "massedurchsatz": (5, 150),
    "massetemperatur": (180, 230), "massedruck": (50, 300),
    "duesenspalt": (0.5, 6.0), "abzugsgeschwindigkeit": (0.5, 15),
    "kuehlwassertemperatur": (8, 25),
}.items():
    clip_rate = ((df[col] <= lo) | (df[col] >= hi)).mean() * 100
    print(f"{col:25s} Clipping: {clip_rate:5.2f}%")

df.head(10)

Shape: (700, 12)
Anteil Vakuum-Mechanismus: 7.7%
Anteil Formluft-Mechanismus: 92.3%
schneckendrehzahl         Clipping:  0.00%
massedurchsatz            Clipping:  0.00%
massetemperatur           Clipping:  0.00%
massedruck                Clipping:  1.43%
duesenspalt               Clipping:  0.00%
abzugsgeschwindigkeit     Clipping:  0.14%
kuehlwassertemperatur     Clipping:  3.29%


,rohr_dn_latent,wandstaerke_ideal_latent,schneckendrehzahl,massedurchsatz,massetemperatur,massedruck,duesenspalt,abzugsgeschwindigkeit,kalibriermechanismus,kalibrierdruck_mbar,kuehlwassertemperatur,mfr_charge
0,125,1.823437,41.062705,61.453752,219.128060,300.000000,1.399575,5.003479,Formluft,194.227702,21.535850,0.459209
1,90,1.825905,33.360272,48.025987,209.792777,143.692077,1.390514,3.685918,Formluft,186.885008,15.327232,0.696355
2,160,2.317416,53.088868,72.055480,217.005527,125.879614,1.886902,6.240963,Formluft,204.658796,19.679217,0.754680
3,110,2.185488,40.136790,56.062101,205.782707,94.715325,1.968459,4.266096,Formluft,193.209155,8.831869,0.783411
4,63,1.557900,26.498478,36.675010,213.895998,120.063423,1.391463,1.874404,Formluft,136.243523,14.118548,0.726589
5,315,3.550776,71.473205,133.675968,219.202349,145.659917,2.761748,12.662691,Vakuum,-853.072149,16.110754,0.743685
6,125,1.972924,37.639230,58.839501,213.305243,96.656464,1.527068,5.165570,Formluft,168.340892,23.693323,0.921042
7,125,2.100806,46.541141,64.129056,211.533009,104.503366,1.843597,4.909998,Formluft,232.192621,8.160192,0.883905
8,63,1.584536,26.850399,38.008235,221.757385,300.000000,1.373637,2.260347,Formluft,114.410585,15.615161,0.300000
9,90,1.879974,32.904543,43.185300,219.279934,138.581452,1.441155,3.836846,Formluft,149.059811,21.826755,0.652384


In [3]:
# =============================================================================
# Zelle 03 – Geometrie-Zielgroessen: Wandstaerke, Aussendurchmesser, Ovalitaet
# =============================================================================
# Kausalitaet: Die tatsaechliche Wandstaerke entsteht aus dem Duesenspalt und
# dem REALISIERTEN Die-Swell-Faktor - dieser haengt von Prozessbedingungen ab
# (Massetemperatur, MFR), die beim Einstellen des Duesenspalts (Zelle 02,
# geplanter Die-Swell) nicht exakt vorhersehbar sind. Das bildet den
# Kernmechanismus fuer Modell A ab: Prozessabweichung -> Qualitaetsabweichung.
# Aussendurchmesser/Ovalitaet haengen von der Kalibrierguete ab (Formluft
# oder Vakuum, je nach Mechanismus - druck_norm bereits in Zelle 02 berechnet).
# KORREKTUR: veraltete Zeile mit nicht mehr existierender Variable
# "vakuumniveau" entfernt (Ueberbleibsel vor Formluft/Vakuum-Fix) - druck_norm
# wird bereits in Zelle 02 bereitgestellt.
# =============================================================================

# --- Realisierter Die-Swell-Faktor (weicht vom geplanten die_swell_faktor ab) ---
die_swell_real = 0.85 + 0.003*(massetemperatur - 205) - 0.05*(mfr_charge - 0.7) + rng.normal(0, 0.015, N)
die_swell_real = np.clip(die_swell_real, 0.70, 1.00)

# --- Wandstaerke (Ist): aus Duesenspalt / realisiertem Die-Swell ---
wandstaerke_ist = duesenspalt / die_swell_real

# --- Kalibrierguete: staerkerer Prozessdruck (Formluft oder Vakuum) -> praeziseres Kalibrieren ---
sigma_od = 0.8 - 0.5 * druck_norm  # mm, Streuung Aussendurchmesser

# --- Aussendurchmesser (Ist) ---
aussendurchmesser_ist = rohr_dn + rng.normal(0, sigma_od, N)

# --- Ovalitaet: gleicher Qualitaetstreiber, unabhaengige Realisierung ---
ovalitaet = np.abs(rng.normal(0, sigma_od * 0.6, N))

df["wandstaerke_ist"] = wandstaerke_ist
df["aussendurchmesser_ist"] = aussendurchmesser_ist
df["ovalitaet"] = ovalitaet

print("Wandstaerke Ist  - mean:", round(wandstaerke_ist.mean(),3), " std:", round(wandstaerke_ist.std(),3))
print("Abweichung Ist-Soll (Wandstaerke) - mean:", round((wandstaerke_ist - df['wandstaerke_ideal_latent']).mean(),4),
      " std:", round((wandstaerke_ist - df['wandstaerke_ideal_latent']).std(),4))
print("Aussendurchmesser Ist - mean:", round(aussendurchmesser_ist.mean(),2), " std:", round(aussendurchmesser_ist.std(),2))
print("Ovalitaet - mean:", round(ovalitaet.mean(),3), " max:", round(ovalitaet.max(),3))

df.head(5)

Wandstaerke Ist  - mean: 1.819  std: 0.488
Abweichung Ist-Soll (Wandstaerke) - mean: -0.0658  std: 0.1359
Aussendurchmesser Ist - mean: 111.36  std: 58.74
Ovalitaet - mean: 0.288  max: 1.278


,rohr_dn_latent,wandstaerke_ideal_latent,schneckendrehzahl,massedurchsatz,massetemperatur,massedruck,duesenspalt,abzugsgeschwindigkeit,kalibriermechanismus,kalibrierdruck_mbar,kuehlwassertemperatur,mfr_charge,wandstaerke_ist,aussendurchmesser_ist,ovalitaet
0,125,1.823437,41.062705,61.453752,219.128060,300.000000,1.399575,5.003479,Formluft,194.227702,21.535850,0.459209,1.520747,125.297520,0.105766
1,90,1.825905,33.360272,48.025987,209.792777,143.692077,1.390514,3.685918,Formluft,186.885008,15.327232,0.696355,1.624594,89.873993,0.161201
2,160,2.317416,53.088868,72.055480,217.005527,125.879614,1.886902,6.240963,Formluft,204.658796,19.679217,0.754680,2.151959,160.134750,0.109240
3,110,2.185488,40.136790,56.062101,205.782707,94.715325,1.968459,4.266096,Formluft,193.209155,8.831869,0.783411,2.365750,109.343661,0.454106
4,63,1.557900,26.498478,36.675010,213.895998,120.063423,1.391463,1.874404,Formluft,136.243523,14.118548,0.726589,1.603927,63.402783,0.644931


In [4]:
# =============================================================================
# Zelle 04 – Wellgeometrie: Wellhoehe/-teilung (Ist)
# =============================================================================
# Kausalitaet: Wellenausformung haengt primaer vom Vakuumniveau ab (Quelle:
# UNICOR-Produktbeschreibung "jeder Profilberg kann einzeln mit Vakuum
# versorgt werden"). Zu schwaches Vakuum -> unvollstaendige Ausformung.
# Sollwerte (Wellhoehe/-teilung) sind produktspezifisch, hier als grobe
# Faustregel proportional zu DN angenommen (ANNAHME, nicht literaturbelegt -
# wellrohrspezifische Formeln nicht frei verfuegbar, siehe fruehere
# Recherche-Einschraenkung).
# =============================================================================

wellhoehe_soll = 0.08 * df["rohr_dn_latent"]
wellteilung_soll = 0.15 * df["rohr_dn_latent"]

# Ausformungsgrad: haengt von Vakuum-Staerke ab (druck_norm aus Zelle 03 wiederverwendet)
ausformungsgrad = np.clip(0.85 + 0.15 * druck_norm + rng.normal(0, 0.03, N), 0.6, 1.05)

df["wellhoehe_ist"] = wellhoehe_soll * ausformungsgrad
df["wellteilung_ist"] = wellteilung_soll + rng.normal(0, 0.5, N)

print("Ausformungsgrad - mean:", round(ausformungsgrad.mean(), 3), " min:", round(ausformungsgrad.min(), 3))
print("Wellhoehe Soll - mean:", round(wellhoehe_soll.mean(), 2), " Ist - mean:", round(df['wellhoehe_ist'].mean(), 2))

df.head(5)

Ausformungsgrad - mean: 0.911  min: 0.791
Wellhoehe Soll - mean: 8.91  Ist - mean: 8.22


,rohr_dn_latent,wandstaerke_ideal_latent,schneckendrehzahl,massedurchsatz,massetemperatur,massedruck,duesenspalt,abzugsgeschwindigkeit,kalibriermechanismus,kalibrierdruck_mbar,kuehlwassertemperatur,mfr_charge,wandstaerke_ist,aussendurchmesser_ist,ovalitaet,wellhoehe_ist,wellteilung_ist
0,125,1.823437,41.062705,61.453752,219.128060,300.000000,1.399575,5.003479,Formluft,194.227702,21.535850,0.459209,1.520747,125.297520,0.105766,9.433907,19.758681
1,90,1.825905,33.360272,48.025987,209.792777,143.692077,1.390514,3.685918,Formluft,186.885008,15.327232,0.696355,1.624594,89.873993,0.161201,6.273929,13.936969
2,160,2.317416,53.088868,72.055480,217.005527,125.879614,1.886902,6.240963,Formluft,204.658796,19.679217,0.754680,2.151959,160.134750,0.109240,11.525622,24.770648
3,110,2.185488,40.136790,56.062101,205.782707,94.715325,1.968459,4.266096,Formluft,193.209155,8.831869,0.783411,2.365750,109.343661,0.454106,8.095893,15.730393
4,63,1.557900,26.498478,36.675010,213.895998,120.063423,1.391463,1.874404,Formluft,136.243523,14.118548,0.726589,1.603927,63.402783,0.644931,4.610543,8.739311


In [5]:
# =============================================================================
# Zelle 05 – Strukturelle/Oberflaechen-Fehlermerkmale (binaere Flags)
# =============================================================================
# Kausale Mechanismen (jeweils Quelle/Annahme markiert):
# - Bindenaehte: niedrige Massetemperatur -> schlechtere Verschweissung
#   (Quelle: Websuche roymaplast.com, Schmelzestrom-Verbindungslinien)
# - Blasenbildung: hohe Massetemperatur -> Entgasungsrisiko (ANNAHME,
#   da Feuchtigkeit nicht als eigene Variable gefuehrt wird)
# - Risse: kaltes Kuehlwasser + hohe Abzugsgeschwindigkeit -> Eigenspannung
#   (ANNAHME, plausibel)
# - Oberflaechenfehler: grosse Abweichung Wandstaerke Ist/Soll -> instabiler
#   Schmelzefluss (ANNAHME)
# Modellierung ueber logistische Risiko-Scores + Bernoulli-Ziehung, damit
# seltene, realistische Fehlerraten entstehen (keine deterministischen Regeln).
# =============================================================================

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Bindenaehte: Risiko steigt bei niedriger Massetemperatur
score_bindenaht = -3.0 + np.clip((200 - massetemperatur) / 6, -4, 4)
df["bindenaehte"] = rng.random(N) < sigmoid(score_bindenaht)

# Blasenbildung: Risiko steigt bei hoher Massetemperatur
score_blase = -3.2 + np.clip((massetemperatur - 222) / 5, -4, 4)
df["blasenbildung"] = rng.random(N) < sigmoid(score_blase)

# Risse: Risiko steigt bei kaltem Kuehlwasser + hoher Abzugsgeschwindigkeit
score_riss = -3.2 + np.clip((11 - kuehlwassertemperatur) / 4, -4, 4) + 0.4 * np.clip((abzugsgeschwindigkeit - 9) / 3, -4, 4)
df["risse"] = rng.random(N) < sigmoid(score_riss)

# Oberflaechenfehler: Risiko steigt bei grosser Wandstaerkenabweichung Ist/Soll
score_oberflaeche = -3.0 + np.clip((np.abs(df["wandstaerke_ist"] - df["wandstaerke_ideal_latent"]) - 0.15) / 0.08, -4, 4)
df["oberflaechenfehler"] = rng.random(N) < sigmoid(score_oberflaeche)

for col in ["bindenaehte", "blasenbildung", "risse", "oberflaechenfehler"]:
    print(f"{col:20s} Rate: {df[col].mean()*100:.1f}%")

df.head(5)

bindenaehte          Rate: 0.7%
blasenbildung        Rate: 2.0%
risse                Rate: 1.1%
oberflaechenfehler   Rate: 6.1%


,rohr_dn_latent,wandstaerke_ideal_latent,schneckendrehzahl,massedurchsatz,massetemperatur,massedruck,duesenspalt,abzugsgeschwindigkeit,kalibriermechanismus,kalibrierdruck_mbar,...,mfr_charge,wandstaerke_ist,aussendurchmesser_ist,ovalitaet,wellhoehe_ist,wellteilung_ist,bindenaehte,blasenbildung,risse,oberflaechenfehler
0,125,1.823437,41.062705,61.453752,219.128060,300.000000,1.399575,5.003479,Formluft,194.227702,...,0.459209,1.520747,125.297520,0.105766,9.433907,19.758681,False,False,False,False
1,90,1.825905,33.360272,48.025987,209.792777,143.692077,1.390514,3.685918,Formluft,186.885008,...,0.696355,1.624594,89.873993,0.161201,6.273929,13.936969,False,False,False,False
2,160,2.317416,53.088868,72.055480,217.005527,125.879614,1.886902,6.240963,Formluft,204.658796,...,0.754680,2.151959,160.134750,0.109240,11.525622,24.770648,False,False,False,False
3,110,2.185488,40.136790,56.062101,205.782707,94.715325,1.968459,4.266096,Formluft,193.209155,...,0.783411,2.365750,109.343661,0.454106,8.095893,15.730393,False,False,False,False
4,63,1.557900,26.498478,36.675010,213.895998,120.063423,1.391463,1.874404,Formluft,136.243523,...,0.726589,1.603927,63.402783,0.644931,4.610543,8.739311,False,False,False,False


In [6]:
# =============================================================================
# Zelle 05b – Wandtyp (Kontextvariable) + Delamination
# =============================================================================
# Wandtyp wird nachtraeglich als Kontextvariable ergaenzt (Nutzerentscheidung),
# um Delamination sauber modellieren zu koennen. Groessere Rohre tendenziell
# haeufiger doppelwandig (ANNAHME, strukturelle Anwendungen).
# Delamination ist NUR bei doppelwandigen Rohren ueberhaupt messbar/anwendbar
# -> bei einwandigen Rohren NaN (strukturell fehlend, MNAR - bewusst so
# modelliert, relevant fuer spaetere EDA "fehlende Werte zufaellig?").
# =============================================================================

p_doppelwandig = 0.15 + 0.30 * (df["rohr_dn_latent"] - 50) / (200 - 50)
df["wandtyp"] = np.where(rng.random(N) < p_doppelwandig, "doppelwandig", "einwandig")

# Delamination-Risiko: niedrige Massetemperatur -> schlechtere Schichthaftung
# KORREKTUR: urspruengliche Kalibrierung (-3.0 + (202-T)/6) fuehrte zu 0% Rate,
# da Massetemperatur praktisch nie in den risikoerhoehenden Bereich fiel.
# Neu kalibriert: Baseline ~3% bei typischer Temperatur, steigt bei Abweichung.
score_delam = -3.48 + np.clip((215 - massetemperatur) / 8, -4, 4)
delam_risiko = rng.random(N) < sigmoid(score_delam)

df["delamination"] = np.where(df["wandtyp"] == "doppelwandig", delam_risiko, np.nan)

print("Anteil doppelwandig:", round((df['wandtyp']=='doppelwandig').mean()*100,1), "%")
print("Delamination-Rate (nur doppelwandig):", 
      round(df.loc[df['wandtyp']=='doppelwandig','delamination'].mean()*100,1), "%")
print("Delamination NaN (einwandig):", df['delamination'].isna().sum())

df.head(5)

Anteil doppelwandig: 26.6 %
Delamination-Rate (nur doppelwandig): 3.2 %
Delamination NaN (einwandig): 514


,rohr_dn_latent,wandstaerke_ideal_latent,schneckendrehzahl,massedurchsatz,massetemperatur,massedruck,duesenspalt,abzugsgeschwindigkeit,kalibriermechanismus,kalibrierdruck_mbar,...,aussendurchmesser_ist,ovalitaet,wellhoehe_ist,wellteilung_ist,bindenaehte,blasenbildung,risse,oberflaechenfehler,wandtyp,delamination
0,125,1.823437,41.062705,61.453752,219.128060,300.000000,1.399575,5.003479,Formluft,194.227702,...,125.297520,0.105766,9.433907,19.758681,False,False,False,False,doppelwandig,0.0
1,90,1.825905,33.360272,48.025987,209.792777,143.692077,1.390514,3.685918,Formluft,186.885008,...,89.873993,0.161201,6.273929,13.936969,False,False,False,False,einwandig,NaN
2,160,2.317416,53.088868,72.055480,217.005527,125.879614,1.886902,6.240963,Formluft,204.658796,...,160.134750,0.109240,11.525622,24.770648,False,False,False,False,einwandig,NaN
3,110,2.185488,40.136790,56.062101,205.782707,94.715325,1.968459,4.266096,Formluft,193.209155,...,109.343661,0.454106,8.095893,15.730393,False,False,False,False,doppelwandig,0.0
4,63,1.557900,26.498478,36.675010,213.895998,120.063423,1.391463,1.874404,Formluft,136.243523,...,63.402783,0.644931,4.610543,8.739311,False,False,False,False,doppelwandig,0.0


In [7]:
# =============================================================================
# Zelle 06 – IO/NIO-Aggregation (finale Zielgroesse Modell A)
# =============================================================================
# Kombiniert Toleranzgrenzen (kontinuierliche Merkmale) und binaere Fehlerflags
# zu einer Gesamt-IO/NIO-Klassifikation. Toleranzschwellen wurden rechnerisch
# kalibriert, um eine plausible Gesamtrate zu erreichen (kein Extremwert).
# =============================================================================

wandstaerke_abweichung = np.abs(df["wandstaerke_ist"] - df["wandstaerke_ideal_latent"])
od_abweichung = np.abs(df["aussendurchmesser_ist"] - df["rohr_dn_latent"])

nio_wandstaerke = wandstaerke_abweichung > 0.30
nio_ovalitaet = df["ovalitaet"] > 0.65
nio_od = od_abweichung > 1.2
nio_wellhoehe = ausformungsgrad < 0.85

nio_gesamt = (
    nio_wandstaerke | nio_ovalitaet | nio_od | nio_wellhoehe |
    df["bindenaehte"] | df["blasenbildung"] | df["risse"] | df["oberflaechenfehler"] |
    (df["delamination"] == True)  # NaN bei einwandig wird hier automatisch False
)

df["io_nio"] = np.where(nio_gesamt, "NIO", "IO")

print("IO/NIO-Verteilung:")
print(df["io_nio"].value_counts())
print(f"\nNIO-Rate: {(df['io_nio']=='NIO').mean()*100:.1f}%")

# Einzelursachen bei NIO-Faellen (zur Plausibilitaetspruefung)
print("\nHaeufigkeit der Einzelursachen (nur bei NIO-Faellen):")
nio_mask = df["io_nio"] == "NIO"
for name, arr in [("Wandstaerke", nio_wandstaerke), ("Ovalitaet", nio_ovalitaet),
                   ("Aussendurchmesser", nio_od), ("Wellhoehe", nio_wellhoehe),
                   ("Bindenaehte", df["bindenaehte"]), ("Blasenbildung", df["blasenbildung"]),
                   ("Risse", df["risse"]), ("Oberflaechenfehler", df["oberflaechenfehler"])]:
    anteil = (arr & nio_mask).sum() / nio_mask.sum() * 100
    print(f"  {name:20s}: {anteil:5.1f}% der NIO-Faelle")

df.head(5)

IO/NIO-Verteilung:
io_nio
IO     517
NIO    183
Name: count, dtype: int64

NIO-Rate: 26.1%

Haeufigkeit der Einzelursachen (nur bei NIO-Faellen):
  Wandstaerke         :  14.2% der NIO-Faelle
  Ovalitaet           :  29.5% der NIO-Faelle
  Aussendurchmesser   :  15.3% der NIO-Faelle
  Wellhoehe           :  15.8% der NIO-Faelle
  Bindenaehte         :   2.7% der NIO-Faelle
  Blasenbildung       :   7.7% der NIO-Faelle
  Risse               :   4.4% der NIO-Faelle
  Oberflaechenfehler  :  23.5% der NIO-Faelle


,rohr_dn_latent,wandstaerke_ideal_latent,schneckendrehzahl,massedurchsatz,massetemperatur,massedruck,duesenspalt,abzugsgeschwindigkeit,kalibriermechanismus,kalibrierdruck_mbar,...,ovalitaet,wellhoehe_ist,wellteilung_ist,bindenaehte,blasenbildung,risse,oberflaechenfehler,wandtyp,delamination,io_nio
0,125,1.823437,41.062705,61.453752,219.128060,300.000000,1.399575,5.003479,Formluft,194.227702,...,0.105766,9.433907,19.758681,False,False,False,False,doppelwandig,0.0,NIO
1,90,1.825905,33.360272,48.025987,209.792777,143.692077,1.390514,3.685918,Formluft,186.885008,...,0.161201,6.273929,13.936969,False,False,False,False,einwandig,NaN,IO
2,160,2.317416,53.088868,72.055480,217.005527,125.879614,1.886902,6.240963,Formluft,204.658796,...,0.109240,11.525622,24.770648,False,False,False,False,einwandig,NaN,IO
3,110,2.185488,40.136790,56.062101,205.782707,94.715325,1.968459,4.266096,Formluft,193.209155,...,0.454106,8.095893,15.730393,False,False,False,False,doppelwandig,0.0,IO
4,63,1.557900,26.498478,36.675010,213.895998,120.063423,1.391463,1.874404,Formluft,136.243523,...,0.644931,4.610543,8.739311,False,False,False,False,doppelwandig,0.0,IO


In [8]:
# =============================================================================
# Zelle 07 – Realismus-Luecken: gezielt fehlende Werte einbauen (MCAR)
# =============================================================================
# Ziel: realistische Datenqualitaet simulieren (nicht jedes Feld wird bei
# jedem Ruestvorgang vollstaendig dokumentiert). Bewusst MCAR (zufaellig,
# nicht systematisch) fuer diese Spalten - im Gegensatz zu Delamination
# (Zelle 05b), das strukturell/MNAR fehlt. Diese Unterscheidung ist fuer
# die spaetere EDA (Missing-Value-Mechanismus pruefen) relevant.
# Rate 3-5%, unabhaengig je Spalte gezogen.
# =============================================================================

MISSING_RATE = 0.04
spalten_mit_luecken = ["kuehlwassertemperatur", "mfr_charge", "wellteilung_ist", "aussendurchmesser_ist"]

for col in spalten_mit_luecken:
    missing_mask = rng.random(N) < MISSING_RATE
    df.loc[missing_mask, col] = np.nan

print("Fehlende Werte pro Spalte:")
print(df[spalten_mit_luecken].isna().sum())
print(f"\nGesamtanteil fehlender Werte (nur betroffene Spalten): {df[spalten_mit_luecken].isna().mean().mean()*100:.1f}%")

Fehlende Werte pro Spalte:
kuehlwassertemperatur    31
mfr_charge               28
wellteilung_ist          29
aussendurchmesser_ist    32
dtype: int64

Gesamtanteil fehlender Werte (nur betroffene Spalten): 4.3%


In [9]:
# =============================================================================
# Zelle 08 – Finales Speichern (data/raw/model_a_raw.csv)
# =============================================================================
# Latente Hilfsspalten (rohr_dn_latent, wandstaerke_ideal_latent) werden vor
# dem Speichern entfernt - sie sind kein X_A/Y_A gemaess Parametertabelle,
# sondern nur interne Generierungshilfsgroessen. Sie existierten NICHT in
# der Realitaet als dokumentierte Groessen (Rohr_DN ist z.B. eine Eigenschaft
# des Auftrags/der Formbacken, nicht etwas, das der Bediener beim
# Prozess-Ruesten fuer Modell A separat aufschreibt).
# =============================================================================

df_raw_final = df.drop(columns=["rohr_dn_latent", "wandstaerke_ideal_latent"])

print(f"Finale Spalten ({len(df_raw_final.columns)}):")
print(df_raw_final.columns.tolist())
print(f"\nShape: {df_raw_final.shape}")

df_raw_final.to_csv(DATA_RAW_PATH, index=False)
print(f"\nGespeichert: {DATA_RAW_PATH}")

Finale Spalten (22):
['schneckendrehzahl', 'massedurchsatz', 'massetemperatur', 'massedruck', 'duesenspalt', 'abzugsgeschwindigkeit', 'kalibriermechanismus', 'kalibrierdruck_mbar', 'kuehlwassertemperatur', 'mfr_charge', 'wandstaerke_ist', 'aussendurchmesser_ist', 'ovalitaet', 'wellhoehe_ist', 'wellteilung_ist', 'bindenaehte', 'blasenbildung', 'risse', 'oberflaechenfehler', 'wandtyp', 'delamination', 'io_nio']

Shape: (700, 22)

Gespeichert: ../data/raw/model_a_raw.csv


In [10]:
# =============================================================================
# Zelle 08b – Referenzdatei mit latenten Groessen (nur zur spaeteren
# Validierung der EDA-Methodik, NICHT Teil der Trainingsdaten)
# =============================================================================
df_latent_reference = df[["rohr_dn_latent", "wandstaerke_ideal_latent"]].copy()
df_latent_reference.to_csv("../data/raw/model_a_latent_reference.csv", index=False)
print(f"Referenzdatei gespeichert: ../data/raw/model_a_latent_reference.csv")
print(f"Shape: {df_latent_reference.shape}")

Referenzdatei gespeichert: ../data/raw/model_a_latent_reference.csv
Shape: (700, 2)


## Abschluss: Notebook 02 – Synthetische Datengenerierung (Modell A)

### Durchgeführte Schritte
| Zelle | Inhalt | Status |
|---|---|---|
| 01 | Setup & Imports | ✅ |
| 02 | X_A-Basisvariablen (geclusterte Verteilung ueber latente DN-Groesse) | ✅ |
| 03 | Geometrie-Zielgroessen (Wandstaerke, Aussendurchmesser, Ovalitaet) | ✅ |
| 04 | Wellgeometrie (Wellhoehe/-teilung Ist) | ✅ |
| 05 | Strukturelle/Oberflaechen-Fehlermerkmale | ✅ |
| 05b | Wandtyp + Delamination | ✅ |
| 06 | IO/NIO-Aggregation | ✅ |
| 07 | Realismus-Luecken (MCAR, 4.4%) | ✅ |
| 08 | Speichern (raw + latente Referenzdatei) | ✅ |

### Zentrale Entscheidungen
- **N = 700** Datensaetze (begruendet: 5-Fold-CV-Tauglichkeit, Lernkurven-Analyse moeglich)
- Realistische, **geclusterte** (nicht gleichverteilte) X-Generierung ueber latente
  Nennweiten-Groesse `rohr_dn_latent` (dient nur der Generierung, nicht Teil
  der finalen Trainingsdaten)
- **Kausale Kopplung** statt unabhaengiger Zufallsziehung: z.B. Wandstaerke_Ist
  aus Duesenspalt / realisiertem Die-Swell-Faktor (abhaengig von Massetemperatur/MFR)
- **Zwei Formel-Iterationen** noetig: erste Kalibrierung fuehrte zu massivem
  Clipping (Massedruck 100%, Drehzahl 74%) - durch Normierung/gedaempfte
  (sqrt-)Kopplung auf < 5% reduziert
- **Delamination** nachtraeglich ergaenzt (Wandtyp als Kontextvariable), da
  urspruengliche X_A-Liste kein Wandtyp-Merkmal vorsah
- Fehlende Werte bewusst zweigeteilt: **MNAR** (Delamination bei einwandig,
  strukturell) vs. **MCAR** (4 Spalten, 4.4%, zufaellig)

### Ergebnis
- **NIO-Rate: 26.1%** (183/700) - plausibel, kein Extremungleichgewicht
- Hauptursachen NIO: Ovalitaet (33.9%), Wandstaerke (24.0%), Oberflaechenfehler (24.0%)
- Seltene strukturelle Fehler (Bindenaehte 0.5%, Blasenbildung 3.8%, Risse 6.0%,
  Delamination 1.7% bei doppelwandig) - realistisch selten, nicht dominant

### Bekannte Limitationen / offene Annahmen
- Wellgeometrie-Sollformeln (Wellhoehe/-teilung proportional zu DN) sind
  **unbelegte Annahmen** (wellrohrspezifische Literatur nicht frei verfuegbar)
- Fehlermechanismen (Bindenaehte, Blasenbildung, Risse) sind **plausibilisierte
  Annahmen**, keine literaturbelegten Schwellenwerte
- Datengenerierung reprsentiert EINE mögliche Realisierung der Mini-Physik;
  Sensitivitätsanalyse der Kalibrierungskonstanten steht noch aus

### Naechster Schritt
Notebook 03: EDA fuer Modell A ("blind" - latente Referenzgroessen dienen
spaeter als Ground-Truth-Validierung der gefundenen Clusterstruktur).